# Kaggle LLM Quantization

Run this notebook on the **Kaggle Jupyter kernel** (Cursor/VS Code → Select Kernel → Existing Jupyter Server).

The Jupyter URL from Kaggle is only a session login for the editor. It is **not** a Hugging Face token and should never be pasted into this notebook or committed.

What this does on the Kaggle machine:
1. Checks GPU / disk
2. Downloads a Hugging Face model into `/kaggle/working`
3. Quantizes with bitsandbytes (4-bit NF4 by default)
4. Smoke-tests generation
5. Saves the quantized weights for download

In [2]:
import os
import shutil
from pathlib import Path

# --- change these ---
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # 3B fits T4/P100 16GB easily; 7B/8B also OK in 4-bit
QUANT_BITS = 4  # 4 or 8
SMOKE_PROMPT = "Explain model quantization in two sentences."
MAX_NEW_TOKENS = 128
DELETE_FP16_AFTER_QUANT = True  # frees /kaggle/working space after saving 4-bit weights

IS_KAGGLE = Path("/kaggle/working").exists()
ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("./output")
HF_HOME = ROOT / "hf"
QUANT_DIR = ROOT / "quantized" / MODEL_ID.replace("/", "__")

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HF_HOME.mkdir(parents=True, exist_ok=True)
QUANT_DIR.mkdir(parents=True, exist_ok=True)

print(f"kaggle={IS_KAGGLE}")
print(f"model={MODEL_ID}")
print(f"quant={QUANT_BITS}-bit")
print(f"hf_home={HF_HOME}")
print(f"save_to={QUANT_DIR}")

kaggle=True
model=Qwen/Qwen2.5-3B-Instruct
quant=4-bit
hf_home=/kaggle/working/hf
save_to=/kaggle/working/quantized/Qwen__Qwen2.5-3B-Instruct


In [3]:
import torch

def disk_gb(path: Path) -> tuple[float, float]:
    usage = shutil.disk_usage(path)
    return usage.free / 1024**3, usage.total / 1024**3

free_gb, total_gb = disk_gb(ROOT)
print(f"disk free {free_gb:.1f} / {total_gb:.1f} GB at {ROOT}")
print(f"cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"gpu={gpu}  vram={vram:.1f} GB")
else:
    raise RuntimeError("No GPU. In Kaggle: Settings → Accelerator → GPU T4/P100, then restart the session.")

if free_gb < 8:
    print("WARNING: less than 8GB free. A 7B download can fail. Clear /kaggle/working or pick a smaller model.")

disk free 19.5 / 19.5 GB at /kaggle/working
cuda=True
gpu=Tesla T4  vram=14.6 GB


In [4]:
%pip install -q -U "transformers>=4.44.0" accelerate bitsandbytes huggingface_hub hf_transfer sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 93.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is n

In [5]:
from huggingface_hub import login, whoami

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not hf_token and IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    info = whoami()
    print(f"logged in as {info.get('name')}")
else:
    print("No HF token. Public models still download. For gated models, add Kaggle secret HF_TOKEN.")

No HF token. Public models still download. For gated models, add Kaggle secret HF_TOKEN.


In [6]:
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_ID} onto the Kaggle machine...")
local_model = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(HF_HOME),
    token=hf_token,
    resume_download=True,
)
print(f"downloaded to {local_model}")
free_gb, _ = disk_gb(ROOT)
print(f"disk free now {free_gb:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

downloaded to /kaggle/working/hf/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1
disk free now 13.7 GB


In [7]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

if QUANT_BITS == 4:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
elif QUANT_BITS == 8:
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
else:
    raise ValueError("QUANT_BITS must be 4 or 8")

tokenizer = AutoTokenizer.from_pretrained(local_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {MODEL_ID} in {QUANT_BITS}-bit...")
model = AutoModelForCausalLM.from_pretrained(
    local_model,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(model.get_memory_footprint() / 1024**3, "GB memory footprint")

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

1.872032642364502 GB memory footprint


In [8]:
messages = [{"role": "user", "content": SMOKE_PROMPT}]
try:
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
except Exception:
    text = SMOKE_PROMPT

inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
print(tokenizer.decode(out[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain model quantization in two sentences.
assistant
Model quantization is the process of reducing the precision of model parameters and activations to smaller values (such as 8-bit integers instead of 32-bit floats), which can significantly reduce the computational resources and memory needed for inference while potentially maintaining performance.


In [9]:
print(f"Saving quantized model to {QUANT_DIR}")
model.save_pretrained(QUANT_DIR)
tokenizer.save_pretrained(QUANT_DIR)
print("saved")

if DELETE_FP16_AFTER_QUANT:
    # Keep only the quantized copy in /kaggle/working to stay under the ~20GB cap.
    snapshots = HF_HOME / "hub"
    if snapshots.exists():
        shutil.rmtree(snapshots, ignore_errors=True)
        print(f"removed original download cache at {snapshots}")

free_gb, _ = disk_gb(ROOT)
print(f"disk free now {free_gb:.1f} GB")
print("Download the folder from the Kaggle notebook Output, or zip it next.")

Saving quantized model to /kaggle/working/quantized/Qwen__Qwen2.5-3B-Instruct


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved
disk free now 11.8 GB
Download the folder from the Kaggle notebook Output, or zip it next.


In [10]:
zip_path = ROOT / f"{QUANT_DIR.name}-bnb{QUANT_BITS}bit"
archive = Path(shutil.make_archive(str(zip_path), "zip", QUANT_DIR))
print(f"archive={archive}")
print(f"size={archive.stat().st_size / 1024**3:.2f} GB")

if IS_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink(archive.name))

archive=/kaggle/working/Qwen__Qwen2.5-3B-Instruct-bnb4bit.zip
size=1.73 GB


/kaggle/working/Qwen__Qwen2.5-3B-Instruct-bnb4bit.zip

## llama.cpp inference (GGUF)

vLLM is dropped. llama.cpp does **not** load the bitsandbytes folder. It needs a **GGUF** file.

This section downloads the official Qwen GGUF `q4_k_m` (same model, llama.cpp’s 4-bit) and runs it with `llama-cpp-python` on the Kaggle T4.

Skip the vLLM cells. Keep the Kaggle kernel selected.

In [16]:
import gc
import os
import shutil
from pathlib import Path

import torch

for _name in ("model", "tokenizer", "llm"):
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
GGUF_DIR = ROOT / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

print("cuda", torch.cuda.is_available(), torch.version.cuda)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("disk free GB", round(shutil.disk_usage(ROOT).free / 1024**3, 1))
print("gguf dir", GGUF_DIR)

cuda True 12.8
Tesla T4
disk free GB 15.8
gguf dir /kaggle/working/gguf


In [17]:
import subprocess
import sys

import torch

cuda = torch.version.cuda or "12.4"
major, minor = cuda.split(".")[:2]
tags = [f"cu{major}{minor}", "cu124", "cu122", "cu121"]
seen = []
for tag in tags:
    if tag in seen:
        continue
    seen.append(tag)
    url = f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"
    print("install llama-cpp-python from", url)
    rc = subprocess.call(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "llama-cpp-python", "--extra-index-url", url]
    )
    if rc == 0:
        print("pip ok", tag)
        break
else:
    print("CUDA wheels failed; installing CPU wheel")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "llama-cpp-python"])

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__)
print("If this cell upgraded packages, restart the kernel, then skip to the download cell.")

install llama-cpp-python from https://abetlen.github.io/llama-cpp-python/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
humming-kernels 0.1.10 requires torch>=2.7, but you have torch 2.6.0 which is incompatible.
nvidia-cutlass-dsl-libs-cu12 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 4.25.9 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.2 which is incompatible.
mistral-common 1.11.7 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.5.2 which is incompatible.
nvidia-cutlass-dsl-libs-base 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 4.25.9 which is incompatible.
nvidia-cutlass-dsl-libs-core 4.6.0 requires protobuf<7,>=6.30.2, but you have protobuf 4.25.9 which is 

pip ok cu128
llama-cpp-python 0.3.35
If this cell upgraded packages, restart the kernel, then skip to the download cell.


In [18]:
from pathlib import Path
from huggingface_hub import hf_hub_download

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
GGUF_DIR = ROOT / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)

GGUF_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"
GGUF_FILE = "qwen2.5-3b-instruct-q4_k_m.gguf"

gguf_path = Path(
    hf_hub_download(
        repo_id=GGUF_REPO,
        filename=GGUF_FILE,
        local_dir=str(GGUF_DIR),
    )
)
print("gguf", gguf_path)
print("size GB", round(gguf_path.stat().st_size / 1024**3, 2))

qwen2.5-3b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 2.10GB            

qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

gguf /kaggle/working/gguf/qwen2.5-3b-instruct-q4_k_m.gguf
size GB 1.96


In [19]:
from pathlib import Path
from llama_cpp import Llama

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
gguf_path = ROOT / "gguf" / "qwen2.5-3b-instruct-q4_k_m.gguf"
if not gguf_path.exists():
    matches = list((ROOT / "gguf").rglob("*.gguf"))
    if not matches:
        raise FileNotFoundError(f"No GGUF at {gguf_path}. Re-run the download cell.")
    gguf_path = matches[0]

llm = Llama(
    model_path=str(gguf_path),
    n_ctx=2048,
    n_gpu_layers=-1,  # offload all layers to the T4; uses CPU if this wheel has no CUDA
    n_threads=4,
    verbose=True,
)
print("loaded", gguf_path)

llama_model_loader: loaded meta data with 26 key-value pairs and 435 tensors from /kaggle/working/gguf/qwen2.5-3b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = qwen2.5-3b-instruct
llama_model_loader: - kv   3:                            general.version str              = v0.1-v0.1
llama_model_loader: - kv   4:                           general.finetune str              = qwen2.5-3b-instruct
llama_model_loader: - kv   5:                         general.size_label str              = 3.4B
llama_model_loader: - kv   6:                          qwen2.block_count u32              = 36
llama_model_loader: - kv  

loaded /kaggle/working/gguf/qwen2.5-3b-instruct-q4_k_m.gguf


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'tokenizer.ggml.add_bos_token': 'false', 'tokenizer.ggml.bos_token_id': '151643', 'general.file_type': '15', 'qwen2.attention.layer_norm_rms_epsilon': '0.000001', 'general.architecture': 'qwen2', 'tokenizer.ggml.padding_token_id': '151643', 'qwen2.embedding_length': '2048', 'tokenizer.ggml.pre': 'qwen2', 'general.name': 'qwen2.5-3b-instruct', 'qwen2.block_count': '36', 'general.version': 'v0.1-v0.1', 'tokenizer.ggml.eos_token_id': '151645', 'qwen2.rope.freq_base': '1000000.000000', 'general.finetune': 'qwen2.5-3b-instruct', 'general.type': 'model', 'general.size_label': '3.4B', 'qwen2.context_length': '32768', 'tokenizer.chat_template': '{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created

In [20]:
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Explain model quantization in two sentences."}],
    max_tokens=128,
    temperature=0.0,
)
print(out["choices"][0]["message"]["content"])

out2 = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Write a Python function that counts tokens naively by splitting on spaces."}],
    max_tokens=128,
    temperature=0.0,
)
print("\n=====\n")
print(out2["choices"][0]["message"]["content"])

llama_perf_context_print:        load time =    1701.06 ms
llama_perf_context_print: prompt eval time =    1700.15 ms /    38 tokens (   44.74 ms per token,    22.35 tokens per second)
llama_perf_context_print:        eval time =    6593.37 ms /    53 runs   (  124.40 ms per token,     8.04 tokens per second)
llama_perf_context_print:       total time =    8346.36 ms /    91 tokens
llama_perf_context_print:    graphs reused =         52
Llama.generate: 24 prefix-match hit, remaining 19 prompt tokens to eval


Model quantization is a technique used to reduce the precision of model parameters and activations from floating-point numbers to lower precision formats like integers, which can significantly reduce the model size and computational cost while maintaining acceptable accuracy, especially for applications requiring low-power or edge computing environments.


llama_perf_context_print:        load time =    1701.06 ms
llama_perf_context_print: prompt eval time =     955.59 ms /    19 tokens (   50.29 ms per token,    19.88 tokens per second)
llama_perf_context_print:        eval time =   15683.17 ms /   127 runs   (  123.49 ms per token,     8.10 tokens per second)
llama_perf_context_print:       total time =   16774.67 ms /   146 tokens
llama_perf_context_print:    graphs reused =        126



=====

Certainly! To count tokens in a string by splitting on spaces, you can use Python's built-in `split` method. Here's a simple Python function that does this:

```python
def count_tokens_naively(text):
    """
    Count the number of tokens in a string by splitting on spaces.

    :param text: The input string to be tokenized.
    :return: The number of tokens.
    """
    # Split the text on spaces and count the number of resulting substrings
    tokens = text.split()
    return len(tokens)

# Example usage:
text = "This is a test string."
token_count =


## Chat frontend

Runs a Gradio page on the Kaggle machine, using the GGUF already loaded in `llm`. After launch, open the **public `*.gradio.live` URL** in your browser. Type a prompt, get a reply.

Keep this cell running and the Kaggle session alive. Stop the cell to shut the UI down.

In [21]:
%pip install -q -U "gradio>=4.44.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [22]:
from pathlib import Path
import gradio as gr
from llama_cpp import Llama

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./output")
GGUF_PATH = ROOT / "gguf" / "qwen2.5-3b-instruct-q4_k_m.gguf"

if "llm" not in globals() or llm is None:
    if not GGUF_PATH.exists():
        matches = list((ROOT / "gguf").rglob("*.gguf"))
        if not matches:
            raise FileNotFoundError("GGUF missing. Re-run the llama.cpp download cell first.")
        GGUF_PATH = matches[0]
    llm = Llama(
        model_path=str(GGUF_PATH),
        n_ctx=2048,
        n_gpu_layers=-1,
        n_threads=4,
        verbose=False,
    )


def _as_messages(message, history):
    msgs = []
    for turn in history or []:
        if isinstance(turn, dict):
            role = turn.get("role", "user")
            content = turn.get("content", "")
            if isinstance(content, list):
                content = "".join(
                    part.get("text", "") if isinstance(part, dict) else str(part) for part in content
                )
            msgs.append({"role": role, "content": content})
        else:
            user, assistant = turn
            msgs.append({"role": "user", "content": user})
            if assistant:
                msgs.append({"role": "assistant", "content": assistant})
    if isinstance(message, dict):
        message = message.get("content", "")
    msgs.append({"role": "user", "content": str(message)})
    return msgs


def chat(message, history):
    acc = ""
    stream = llm.create_chat_completion(
        messages=_as_messages(message, history),
        max_tokens=512,
        temperature=0.7,
        stream=True,
    )
    for chunk in stream:
        delta = (chunk.get("choices") or [{}])[0].get("delta") or {}
        piece = delta.get("content") or ""
        if piece:
            acc += piece
            yield acc


demo = gr.ChatInterface(
    fn=chat,
    title="Qwen2.5-3B Instruct (Q4_K_M)",
    description="llama.cpp on Kaggle. Type a prompt and wait for the streamed reply.",
    examples=[
        "Explain model quantization in two sentences.",
        "Write a Python function that counts tokens by splitting on spaces.",
        "Give me a 5-step plan to deploy a small LLM.",
    ],
)

print("Launching. Use the public https://*.gradio.live link, not localhost.")
demo.launch(share=True, inline=False, prevent_thread_lock=True)

Launching. Use the public https://*.gradio.live link, not localhost.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c8468a0d856933accf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Llama.generate: 24 prefix-match hit, remaining 9 prompt tokens to eval
llama_perf_context_print:        load time =    1701.06 ms
llama_perf_context_print: prompt eval time =     629.61 ms /     9 tokens (   69.96 ms per token,    14.29 tokens per second)
llama_perf_context_print:        eval time =    3235.47 ms /    23 runs   (  140.67 ms per token,     7.11 tokens per second)
llama_perf_context_print:       total time =    3935.61 ms /    32 tokens
llama_perf_context_print:    graphs reused =         22
Llama.generate: 56 prefix-match hit, remaining 14 prompt tokens to eval
Llama.generate: 183 prefix-match hit, remaining 16 prompt tokens to eval
llama_perf_context_print:        load time =    1701.06 ms
llama_perf_context_print: prompt eval time =     735.10 ms /    16 tokens (   45.94 ms per token,    21.77 tokens per second)
llama_perf_context_print:        eval time =   14100.71 ms /    99 runs   (  142.43 ms per token,     7.02 tokens per second)
llama_perf_context_print:       